# Train BERT Text Classifier from JSONL

Train BERT model using OCR texts from JSONL file.
- Reads from `ocr_texts.backup.jsonl`
- Extracts text and labels
- Trains incrementally in chunks
- Saves model after each chunk


In [20]:
import json
import random
from pathlib import Path
from dataclasses import dataclass
from typing import List

import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

try:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
except ImportError:
    raise RuntimeError("Missing transformers. Install: pip install -U transformers")

CLASS_NAMES = ["not_signed", "signed"]

print("✓ Imports successful")


✓ Imports successful


In [21]:
# Configuration
jsonl_file = "/home/ram-sthapit/programming/Signature Verification/export_images/train/ocr_texts2.jsonl"
out_dir = "/home/ram-sthapit/programming/Signature Verification/text_model_final"
model_name = "distilbert-base-uncased"

# Training parameters
chunk_size = 1000
epochs = 3
batch_size = 8
lr = 2e-5
val_split = 0.1
max_length = 256
seed = 42

# Set random seeds
random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Output directory: {out_dir}")
print(f"Model: {model_name}")


Device: cuda
Output directory: /home/ram-sthapit/programming/Signature Verification/text_model_final
Model: distilbert-base-uncased


In [22]:
@dataclass(frozen=True)
class Example:
    text: str
    label: int

# Read JSONL file
print(f"Reading JSONL file: {jsonl_file}")
examples = []
with open(jsonl_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading"):
        line = line.strip()
        if line:
            record = json.loads(line)
            text = record.get('text', '')
            label = int(record.get('label', 0))
            examples.append(Example(text=text, label=label))

print(f"✓ Loaded {len(examples)} examples")
print(f"  Label 1 (signed): {sum(1 for e in examples if e.label == 1)}")
print(f"  Label 0 (not_signed): {sum(1 for e in examples if e.label == 0)}")


Reading JSONL file: /home/ram-sthapit/programming/Signature Verification/export_images/train/ocr_texts2.jsonl


Loading: 10000it [00:00, 50785.75it/s]

✓ Loaded 10000 examples
  Label 1 (signed): 1200
  Label 0 (not_signed): 8800


In [23]:
class TextDataset(Dataset):
    def __init__(self, examples: List[Example]):
        self.examples = examples

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int) -> Example:
        return self.examples[idx]

def make_collate(tokenizer, max_length: int):
    def collate(batch: List[Example]):
        texts = [b.text for b in batch]
        labels = torch.tensor([b.label for b in batch], dtype=torch.long)
        enc = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        return enc, labels
    return collate

print("✓ Dataset classes defined")


✓ Dataset classes defined


In [24]:
# Initialize model
out_dir_path = Path(out_dir)
out_dir_path.mkdir(parents=True, exist_ok=True)

if (out_dir_path / "config.json").exists():
    print(f"Loading existing model from {out_dir}...")
    tokenizer = AutoTokenizer.from_pretrained(str(out_dir_path), use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(str(out_dir_path)).to(device)
else:
    print(f"Creating new model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss()

print("✓ Model initialized")


Creating new model: distilbert-base-uncased


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Model initialized


In [25]:
def training_epoch(epoch: int, model, train_loader, optimizer, criterion, device, chunk_num: int, total_chunks: int):
    """Train for one epoch."""
    model.train()
    desc = f"Chunk {chunk_num}/{total_chunks} - Epoch {epoch}"
    
    for inputs, targets in tqdm(train_loader, desc=desc, leave=False):
        inputs = {k: v.to(device, non_blocking=False) for k, v in inputs.items()}
        targets = targets.to(device, non_blocking=False)
        
        optimizer.zero_grad()
        outputs = model(**inputs).logits
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

@torch.no_grad()
def evaluate(model, val_loader, device) -> float:
    """Evaluate model."""
    model.eval()
    correct = 0
    total = 0
    
    for inputs, targets in val_loader:
        inputs = {k: v.to(device, non_blocking=False) for k, v in inputs.items()}
        targets = targets.to(device, non_blocking=False)
        
        outputs = model(**inputs).logits
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    return 100.0 * correct / max(1, total)

print("✓ Training functions defined")


✓ Training functions defined


In [26]:
def train_chunk(
    examples_chunk: List[Example],
    model,
    tokenizer,
    optimizer,
    criterion,
    device,
    epochs: int,
    batch_size: int,
    val_split: float,
    max_length: int,
    chunk_num: int,
    total_chunks: int,
):
    """Train on a chunk of data."""
    # Shuffle examples
    examples_shuffled = examples_chunk.copy()
    random.shuffle(examples_shuffled)
    
    # Train/val split
    val_n = max(1, int(len(examples_shuffled) * val_split))
    train_ex, val_ex = examples_shuffled[:-val_n], examples_shuffled[-val_n:]
    print(f"Chunk {chunk_num}: train={len(train_ex)}, val={len(val_ex)}")
    
    # DataLoaders
    collate_fn = make_collate(tokenizer, max_length=max_length)
    train_loader = DataLoader(
        TextDataset(train_ex), batch_size=batch_size, shuffle=True,
        collate_fn=collate_fn, pin_memory=False, num_workers=0
    )
    val_loader = DataLoader(
        TextDataset(val_ex), batch_size=max(1, batch_size // 2), shuffle=False,
        collate_fn=collate_fn, pin_memory=False, num_workers=0
    )
    
    # Training
    best_acc = -1.0
    for epoch in range(1, epochs + 1):
        training_epoch(epoch, model, train_loader, optimizer, criterion, device, chunk_num, total_chunks)
        
        if device.type == "cuda":
            torch.cuda.empty_cache()
        
        acc = evaluate(model, val_loader, device)
        print(f"Chunk {chunk_num}/{total_chunks} - Epoch {epoch}: val_acc={acc:.2f}%")
        
        if acc > best_acc:
            best_acc = acc
    
    return best_acc

print("✓ Chunk training function defined")


✓ Chunk training function defined


In [27]:
# Split examples into chunks
chunks = [examples[i:i + chunk_size] for i in range(0, len(examples), chunk_size)]
print(f"Total samples: {len(examples)}, Chunks: {len(chunks)}\n")

# Train on each chunk
best_acc = -1.0
for chunk_num, examples_chunk in enumerate(chunks, 1):
    print(f"\n{'='*60}")
    print(f"Training on chunk {chunk_num}/{len(chunks)} ({len(examples_chunk)} samples)")
    print(f"{'='*60}\n")
    
    if device.type == "cuda":
        torch.cuda.empty_cache()
    
    acc = train_chunk(
        examples_chunk, model, tokenizer, optimizer, criterion, device,
        epochs, batch_size, val_split, max_length, chunk_num, len(chunks)
    )
    
    # Save after each chunk
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    print(f"✓ Saved model (val_acc={acc:.2f}%) -> {out_dir}")
    
    if device.type == "cuda":
        torch.cuda.empty_cache()
    
    if acc > best_acc:
        best_acc = acc

print(f"\n{'='*60}")
print(f"Training complete! Processed {len(examples)} samples in {len(chunks)} chunks")
print(f"Best validation accuracy: {best_acc:.2f}%")
print(f"Model saved to: {out_dir}")
print(f"{'='*60}")


Total samples: 10000, Chunks: 10


Training on chunk 1/10 (1000 samples)

Chunk 1: train=900, val=100


Chunk 1/10 - Epoch 1: val_acc=97.00%


Chunk 1/10 - Epoch 2: val_acc=98.00%


Chunk 1/10 - Epoch 3: val_acc=99.00%
✓ Saved model (val_acc=99.00%) -> /home/ram-sthapit/programming/Signature Verification/text_model_final

Training on chunk 2/10 (1000 samples)

Chunk 2: train=900, val=100


Chunk 2/10 - Epoch 1: val_acc=98.00%


...

Chunk 9/10 - Epoch 2: val_acc=100.00%


Chunk 9/10 - Epoch 3: val_acc=100.00%
✓ Saved model (val_acc=100.00%) -> /home/ram-sthapit/programming/Signature Verification/text_model_final

Training on chunk 10/10 (1000 samples)

Chunk 10: train=900, val=100


Chunk 10/10 - Epoch 1: val_acc=99.00%


Chunk 10/10 - Epoch 2: val_acc=99.00%


Chunk 10/10 - Epoch 3: val_acc=99.00%
✓ Saved model (val_acc=99.00%) -> /home/ram-sthapit/programming/Signature Verification/text_model_final

Training complete! Processed 10000 samples in 10 chunks
Best validation accuracy: 100.00%
Model saved to: /home/ram-sthapit/programming/Signature Verification/text_model_final
